# Supermarket Sales ETL Project
Complete ETL solution using Kaggle dataset, Python, Pandas and SQLite.

## 1. Data Extraction (Kaggle API)
Install dependencies first:
```bash
pip install kaggle pandas sqlalchemy
```
Place kaggle.json in ~/.kaggle/ before running.

In [ ]:
from kaggle.api.kaggle_api_extended import KaggleApi
api = KaggleApi()
api.authenticate()
api.dataset_download_files('faresashraf1001/supermarket-sales', path='data', unzip=True)
print('Dataset downloaded')

## 2. Schema Design (Star Schema)

### Dimension Tables
- dim_product → branch, city, product_line
- dim_customer → customer_type, gender, payment

### Fact Table
- fact_sales → quantity, price, tax, total, date/time

In [ ]:
import pandas as pd
import sqlite3

df = pd.read_csv('data/supermarket_sales.csv')

# Product dimension
dim_product = df[['Branch','City','Product line']].drop_duplicates().reset_index(drop=True)
dim_product['product_id'] = dim_product.index + 1

# Customer dimension
dim_customer = df[['Customer type','Gender','Payment']].drop_duplicates().reset_index(drop=True)
dim_customer['customer_id'] = dim_customer.index + 1

# Fact table
fact = df.merge(dim_product,on=['Branch','City','Product line'])\
         .merge(dim_customer,on=['Customer type','Gender','Payment'])

fact_sales = fact[['product_id','customer_id','Date','Time','Quantity','Unit price','Tax 5%','Total']]
fact_sales.columns=['product_id','customer_id','date','time','quantity','unit_price','tax','total']

conn = sqlite3.connect('sales_dw.db')
dim_product.to_sql('dim_product',conn,if_exists='replace',index=False)
dim_customer.to_sql('dim_customer',conn,if_exists='replace',index=False)
fact_sales.to_sql('fact_sales',conn,if_exists='replace',index=False)
conn.close()
print('Loaded into SQLite')

## 3. ER Model Summary
- fact_sales links dim_product and dim_customer via foreign keys.
- Star schema optimized for analytics/reporting.